# 因子生成过程演示 Notebook

本 Notebook 用于可视化 `factor_engine` 中因子的**构建 -> 编译 -> 执行 -> 结果检查**过程，并覆盖 `fundamentals` 与 `us_stocks_sip` 的数据类别。

In [1]:
from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 180)

print('project_root =', PROJECT_ROOT)

project_root = /home/yluel/share/projects/factor_engine


In [2]:
from backend.pandas_backend import PandasBackend
from runtime.engine import FactorEngine
from runtime.real_data_factor_smoke import DATASET_SPECS, _build_factors, run_dataset_factor_smoke
from storage.datasource import DataSource

spec_df = pd.DataFrame([
    {
        'dataset': s.name,
        'timestamp_col': s.timestamp_col,
        'instrument_col': s.instrument_col,
        'columns': ', '.join(s.candidate_columns),
        'timestamp_unit': s.timestamp_unit or '-'
    }
    for s in DATASET_SPECS
])

print('dataset_class_count =', len(spec_df))
display(spec_df)

dataset_class_count = 11


,dataset,timestamp_col,instrument_col,columns,timestamp_unit
0,fundamentals/balance_sheet,period_end,tickers,"total_assets, total_liabilities, total_equity",-
1,fundamentals/cash_flow_statement,period_end,tickers,"net_cash_from_operating_activities, net_cash_f...",-
2,fundamentals/financials_ratios,date,ticker,"price_to_earnings, return_on_equity, debt_to_e...",-
3,fundamentals/income_statement,period_end,tickers,"revenue, operating_income, net_income_loss_att...",-
4,fundamentals/short_interest,settlement_date,ticker,"short_interest, days_to_cover, avg_daily_volume",-
5,fundamentals/short_volume,date,ticker,"short_volume, total_volume, short_volume_ratio",-
6,fundamentals/stocks_floats,effective_date,ticker,"free_float, free_float_percent, outstanding_sh...",-
7,us_stocks_sip/day_aggs_v1,window_start,ticker,"close, volume, transactions",ns
8,us_stocks_sip/minute_aggs_v1,window_start,ticker,"close, volume, transactions",ns
9,us_stocks_sip/quotes_v1,sip_timestamp,ticker,"bid_price, ask_price, bid_size",ns


## 1) 用合成数据演示：单类数据的因子生成全过程

这里先不依赖真实 parquet，使用与真实字段一致的列名构造示例数据，便于稳定演示因子链路。

In [3]:
@dataclass
class InMemorySeriesSource(DataSource):
    data: dict[str, pd.Series]

    def load_column(self, name: str):
        return self.data[name]


def build_mock_series(columns: tuple[str, ...]) -> dict[str, pd.Series]:
    ts = pd.date_range('2024-01-01', periods=10, freq='D')
    instruments = ['AAA', 'BBB', 'CCC', 'DDD']
    idx = pd.MultiIndex.from_product([ts, instruments], names=['timestamp', 'instrument'])

    out: dict[str, pd.Series] = {}
    for i, name in enumerate(columns):
        values = [(j + 1) * (i + 1) + (k + 1) * 0.2 for j in range(len(ts)) for k in range(len(instruments))]
        out[name] = pd.Series(values, index=idx)
    return out

target_spec = DATASET_SPECS[0]  # fundamentals/balance_sheet
mock_data = build_mock_series(target_spec.candidate_columns)
source = InMemorySeriesSource(data=mock_data)
engine = FactorEngine(backend=PandasBackend(), data_source=source)
factors = _build_factors(list(target_spec.candidate_columns), prefix=target_spec.name.replace('/', '_'))

print('target_dataset =', target_spec.name)
print('factor_count =', len(factors))
for f in factors:
    print('-', f.name, '| expr =', f.expr)

target_dataset = fundamentals/balance_sheet
factor_count = 4
- fundamentals_balance_sheet_rank_total_assets | expr = Rank(child=ColumnRef(name='total_assets'))
- fundamentals_balance_sheet_zscore_total_assets | expr = ZScore(child=ColumnRef(name='total_assets'))
- fundamentals_balance_sheet_tsmean3_rank_total_assets | expr = Rank(child=TsMean(child=ColumnRef(name='total_assets'), window=3, min_periods=None))
- fundamentals_balance_sheet_spread_total_assets_vs_total_liabilities | expr = ZScore(child=Div(left=ColumnRef(name='total_assets'), right=Add(left=ColumnRef(name='total_liabilities'), right=Literal(value=1e-09))))


In [4]:
compiled_rows = []
for f in factors:
    plan, analysis = engine.compile(f)
    compiled_rows.append({
        'factor': f.name,
        'lookback': analysis.lookback,
        'ir_root': type(analysis.ir).__name__,
        'plan_root_op': plan.op,
    })

compiled_df = pd.DataFrame(compiled_rows)
display(compiled_df)

,factor,lookback,ir_root,plan_root_op
0,fundamentals_balance_sheet_rank_total_assets,0,IRNode,rank
1,fundamentals_balance_sheet_zscore_total_assets,0,IRNode,zscore
2,fundamentals_balance_sheet_tsmean3_rank_total_...,3,IRNode,rank
3,fundamentals_balance_sheet_spread_total_assets...,0,IRNode,zscore


In [5]:
run_rows = []
result_frames = []

for f in factors:
    out = engine.run(f)
    s = out['result'].sort_index()

    run_rows.append({
        'factor': f.name,
        'rows': int(s.shape[0]),
        'non_nan_ratio': float(s.notna().mean()),
        'lookback': int(out['analysis'].lookback),
    })

    preview = s.dropna().groupby(level='instrument').tail(2).rename('value').to_frame()
    preview['factor'] = f.name
    result_frames.append(preview.reset_index())

run_df = pd.DataFrame(run_rows)
preview_df = pd.concat(result_frames, ignore_index=True) if result_frames else pd.DataFrame()

display(run_df)
display(preview_df.head(20))

,factor,rows,non_nan_ratio,lookback
0,fundamentals_balance_sheet_rank_total_assets,40,1.0,0
1,fundamentals_balance_sheet_zscore_total_assets,40,1.0,0
2,fundamentals_balance_sheet_tsmean3_rank_total_...,40,0.8,3
3,fundamentals_balance_sheet_spread_total_assets...,40,1.0,0


,timestamp,instrument,value,factor
0,2024-01-09,AAA,0.250000,fundamentals_balance_sheet_rank_total_assets
1,2024-01-09,BBB,0.500000,fundamentals_balance_sheet_rank_total_assets
2,2024-01-09,CCC,0.750000,fundamentals_balance_sheet_rank_total_assets
3,2024-01-09,DDD,1.000000,fundamentals_balance_sheet_rank_total_assets
4,2024-01-10,AAA,0.250000,fundamentals_balance_sheet_rank_total_assets
5,2024-01-10,BBB,0.500000,fundamentals_balance_sheet_rank_total_assets
6,2024-01-10,CCC,0.750000,fundamentals_balance_sheet_rank_total_assets
7,2024-01-10,DDD,1.000000,fundamentals_balance_sheet_rank_total_assets
8,2024-01-09,AAA,-1.161895,fundamentals_balance_sheet_zscore_total_assets
9,2024-01-09,BBB,-0.387298,fundamentals_balance_sheet_zscore_total_assets


In [8]:
# 展示每个因子的计算结果明细（按因子逐个输出）
factor_value_tables = {}

for f in factors:
    out = engine.run(f)
    s = out['result'].sort_index().rename('factor_value')
    table = s.dropna().to_frame().reset_index()
    factor_value_tables[f.name] = table

for factor_name, table in factor_value_tables.items():
    print(f"\n=== {factor_name} ===")
    print('rows =', len(table), '| non_nan_ratio =', round(table['factor_value'].notna().mean(), 4))
    # 每个因子展示前 12 条结果，便于快速核对
    display(table.head(12))


=== us_stocks_sip_trades_v1_rank_price ===
rows = 40 | non_nan_ratio = 1.0


,timestamp,instrument,factor_value
0,2024-01-01,AAA,0.25
1,2024-01-01,BBB,0.50
2,2024-01-01,CCC,0.75
3,2024-01-01,DDD,1.00
4,2024-01-02,AAA,0.25
5,2024-01-02,BBB,0.50
6,2024-01-02,CCC,0.75
7,2024-01-02,DDD,1.00
8,2024-01-03,AAA,0.25
9,2024-01-03,BBB,0.50



=== us_stocks_sip_trades_v1_zscore_price ===
rows = 40 | non_nan_ratio = 1.0


,timestamp,instrument,factor_value
0,2024-01-01,AAA,-1.161895
1,2024-01-01,BBB,-0.387298
2,2024-01-01,CCC,0.387298
3,2024-01-01,DDD,1.161895
4,2024-01-02,AAA,-1.161895
5,2024-01-02,BBB,-0.387298
6,2024-01-02,CCC,0.387298
7,2024-01-02,DDD,1.161895
8,2024-01-03,AAA,-1.161895
9,2024-01-03,BBB,-0.387298



=== us_stocks_sip_trades_v1_tsmean3_rank_price ===
rows = 32 | non_nan_ratio = 1.0


,timestamp,instrument,factor_value
0,2024-01-03,AAA,0.25
1,2024-01-03,BBB,0.50
2,2024-01-03,CCC,0.75
3,2024-01-03,DDD,1.00
4,2024-01-04,AAA,0.25
5,2024-01-04,BBB,0.50
6,2024-01-04,CCC,0.75
7,2024-01-04,DDD,1.00
8,2024-01-05,AAA,0.25
9,2024-01-05,BBB,0.50



=== us_stocks_sip_trades_v1_spread_price_vs_size ===
rows = 40 | non_nan_ratio = 1.0


,timestamp,instrument,factor_value
0,2024-01-01,AAA,-1.222381
1,2024-01-01,BBB,-0.319852
2,2024-01-01,CCC,0.443826
3,2024-01-01,DDD,1.098407
4,2024-01-02,AAA,-1.195865
5,2024-01-02,BBB,-0.351185
6,2024-01-02,CCC,0.420045
7,2024-01-02,DDD,1.127005
8,2024-01-03,AAA,-1.185510
9,2024-01-03,BBB,-0.362657


## 2) 批量演示 fundamentals + us_stocks_sip 全类别（合成数据）

In [6]:
summary_rows = []

for spec in DATASET_SPECS:
    data = build_mock_series(spec.candidate_columns)
    source = InMemorySeriesSource(data=data)
    engine = FactorEngine(backend=PandasBackend(), data_source=source)
    factors = _build_factors(list(spec.candidate_columns), prefix=spec.name.replace('/', '_'))

    ok_count = 0
    for f in factors:
        out = engine.run(f)
        s = out['result']
        ok_count += int(s.notna().any())

    summary_rows.append({
        'dataset': spec.name,
        'factor_count': len(factors),
        'factor_with_non_nan': ok_count,
        'columns': ', '.join(spec.candidate_columns),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('dataset').reset_index(drop=True)
display(summary_df)

,dataset,factor_count,factor_with_non_nan,columns
0,fundamentals/balance_sheet,4,4,"total_assets, total_liabilities, total_equity"
1,fundamentals/cash_flow_statement,4,4,"net_cash_from_operating_activities, net_cash_f..."
2,fundamentals/financials_ratios,4,4,"price_to_earnings, return_on_equity, debt_to_e..."
3,fundamentals/income_statement,4,4,"revenue, operating_income, net_income_loss_att..."
4,fundamentals/short_interest,4,4,"short_interest, days_to_cover, avg_daily_volume"
5,fundamentals/short_volume,4,4,"short_volume, total_volume, short_volume_ratio"
6,fundamentals/stocks_floats,4,4,"free_float, free_float_percent, outstanding_sh..."
7,us_stocks_sip/day_aggs_v1,4,4,"close, volume, transactions"
8,us_stocks_sip/minute_aggs_v1,4,4,"close, volume, transactions"
9,us_stocks_sip/quotes_v1,4,4,"bid_price, ask_price, bid_size"


## 3) 可选：真实 parquet 运行（受环境/pyarrow 兼容性影响）

如果你的运行时对某些 parquet 文件存在解码兼容问题，本节会显示错误信息，但不影响前面的因子生成过程演示。

In [7]:
MASSIVE_ROOT = Path('/home/yluel/share/projects/massive_parquet')

real_specs = [s for s in DATASET_SPECS if s.name in ('fundamentals/financials_ratios', 'us_stocks_sip/day_aggs_v1')]
real_rows = []

for spec in real_specs:
    report = run_dataset_factor_smoke(MASSIVE_ROOT, spec, max_files=1)
    ok = sum(1 for item in report['results'] if item['ok'])
    fail = sum(1 for item in report['results'] if not item['ok'])
    first_error = next((item['error'] for item in report['results'] if not item['ok']), '')

    real_rows.append({
        'dataset': spec.name,
        'ok_factors': ok,
        'failed_factors': fail,
        'first_error': first_error[:160],
    })

real_df = pd.DataFrame(real_rows)
display(real_df)

,dataset,ok_factors,failed_factors,first_error
0,fundamentals/financials_ratios,4,0,
1,us_stocks_sip/day_aggs_v1,4,0,


## 配置驱动因子演示

下面通过读取 `examples/configs/` 下的 11 个 YAML 配置文件，以 **配置驱动** 方式对每类数据集跑出因子结果。

配置格式：
```yaml
factor:
  name: ...
  expr: ...          # DSL 表达式，如 rank(ts_mean(col("close"), 3))
  freq: 1d
  description: ...

data_source:
  type: multi_parquet
  root: /path/to/dataset
  timestamp_col: ...
  instrument_col: ...
  max_files: 3
  start_date: "2023-01-01"   # 行级时间过滤：左闭
  end_date:   "2024-12-31"   # 行级时间过滤：右闭

backend:
  type: pandas
```

> **关于时间范围**：`start_date` / `end_date` 在加载列数据后做**行级过滤**，确保因子只在指定窗口内计算，避免引入未来数据或加载全量历史。`parquet_kline` 类型在此基础上还额外支持按文件名过滤，进一步减少 I/O。

In [2]:
import yaml
from runtime.engine import FactorEngine

CONFIGS_DIR = PROJECT_ROOT / "examples" / "configs"
config_files = sorted(CONFIGS_DIR.glob("*.yaml"))
print(f"找到 {len(config_files)} 个配置文件:")
for f in config_files:
    print(f"  {f.name}")

找到 11 个配置文件:
  fundamentals_balance_sheet.yaml
  fundamentals_cash_flow_statement.yaml
  fundamentals_financials_ratios.yaml
  fundamentals_income_statement.yaml
  fundamentals_short_interest.yaml
  fundamentals_short_volume.yaml
  fundamentals_stocks_floats.yaml
  us_stocks_sip_day_aggs_v1.yaml
  us_stocks_sip_minute_aggs_v1.yaml
  us_stocks_sip_quotes_v1.yaml
  us_stocks_sip_trades_v1.yaml


In [7]:
config_results = []

for cfg_path in config_files:
    cfg_text = yaml.safe_load(cfg_path.read_text())
    factor_cfg = cfg_text.get("factor", {})
    ds_cfg = cfg_text.get("data_source", {})
    factor_name = factor_cfg.get("name", cfg_path.stem)
    description = factor_cfg.get("description", "")

    try:
        engine, factor, config = FactorEngine.from_config(cfg_path)
        out = engine.run(factor)
        s = out["result"].dropna()
        config_results.append({
            "config_file": cfg_path.name,
            "factor_name": factor_name,
            "description": description,
            "data_type": ds_cfg.get("root", "").replace("/home/yluel/share/projects/massive_parquet/", ""),
            "expr": factor_cfg.get("expr", ""),
            "rows": len(s),
            "non_nan_ratio": round(out["result"].notna().mean(), 4),
            "status": "OK",
        })
    except Exception as exc:
        config_results.append({
            "config_file": cfg_path.name,
            "factor_name": factor_name,
            "description": description,
            "data_type": ds_cfg.get("root", "").replace("/home/yluel/share/projects/massive_parquet/", ""),
            "expr": factor_cfg.get("expr", ""),
            "rows": 0,
            "non_nan_ratio": 0.0,
            "status": f"ERROR: {exc}",
        })

summary_df = pd.DataFrame(config_results)
display(summary_df[["factor_name", "data_type", "expr", "rows", "non_nan_ratio", "status"]])

,factor_name,data_type,expr,rows,non_nan_ratio,status
0,balance_sheet_rank_total_assets,fundamentals/balance_sheet,"rank(col(""total_assets""))",29002,0.9994,OK
1,cash_flow_zscore_operating,fundamentals/cash_flow_statement,"zscore(col(""net_cash_from_operating_activities""))",51217,0.9590,OK
2,financials_ratios_rank_pe,fundamentals/financials_ratios,"rank(col(""price_to_earnings""))",2274,0.4374,OK
3,income_statement_zscore_revenue,fundamentals/income_statement,"zscore(col(""revenue""))",50898,0.9549,OK
4,short_interest_rank_days_to_cover,fundamentals/short_interest,"rank(col(""days_to_cover""))",720962,1.0000,OK
5,short_volume_zscore_ratio,fundamentals/short_volume,"zscore(col(""short_volume_ratio""))",4131048,1.0000,OK
6,stocks_floats_zscore_free_float_pct,fundamentals/stocks_floats,"zscore(col(""free_float_percent""))",6327,0.9932,OK
7,day_aggs_rank_ts_mean_close_3,us_stocks_sip/day_aggs_v1,"rank(ts_mean(col(""close""), 3))",7197,0.3128,OK
8,minute_aggs_rank_ts_mean_close_5,us_stocks_sip/minute_aggs_v1,"rank(ts_mean(col(""close""), 5))",1649645,0.9818,OK
9,quotes_zscore_bid_ask_spread,us_stocks_sip/quotes_v1,"zscore(col(""bid_price"") / (col(""ask_price"") + ...",18386837,1.0000,OK


In [8]:
# 逐个展示成功因子的样本数据（前 12 行）
ok_results = [(r, cfg_path) for r, cfg_path in zip(config_results, sorted(CONFIGS_DIR.glob("*.yaml"))) if r["status"] == "OK"]

for r, cfg_path in ok_results:
    engine, factor, config = FactorEngine.from_config(cfg_path)
    out = engine.run(factor)
    s = out["result"].dropna().sort_index().rename("factor_value")
    table = s.to_frame().reset_index()
    print(f"\n=== {r['factor_name']} ===")
    print(f"描述: {r['description']}")
    print(f"行数: {len(table)}  有效率: {r['non_nan_ratio']}")
    display(table.head(12))


=== balance_sheet_rank_total_assets ===
描述: 资产负债表 - 总资产截面排名，越高代表规模越大
行数: 29002  有效率: 0.9994


,timestamp,instrument,factor_value
0,2009-04-30,['WMT'],1.0
1,2009-05-02,['GME' 'GMEw' 'RRD'],0.2
2,2009-05-02,['GPS'],0.6
3,2009-05-02,['JCP'],0.8
4,2009-05-02,['JWN'],0.3
5,2009-05-02,['KSS'],0.7
6,2009-05-02,['LTD'],0.4
7,2009-05-02,['M'],0.9
8,2009-05-02,['ROST'],0.1
9,2009-05-02,['TGT'],1.0



=== cash_flow_zscore_operating ===
描述: 现金流量表 - 经营活动现金流截面标准化，正值代表现金流强劲
行数: 51217  有效率: 0.959


,timestamp,instrument,factor_value
0,2009-04-30,['ADSK'],-0.594730
1,2009-04-30,['CRM'],-0.559794
2,2009-04-30,['WMT'],1.154524
3,2009-05-02,['GME' 'GMEw' 'RRD'],-1.372270
4,2009-05-02,['GPS'],-0.015861
5,2009-05-02,['JCP'],-0.416398
6,2009-05-02,['JWN'],-0.007216
7,2009-05-02,['KSS'],0.537399
8,2009-05-02,['LTD'],-0.937961
9,2009-05-02,['M'],-0.606581



=== financials_ratios_rank_pe ===
描述: 财务比率 - PE 截面排名，排名越低代表估值越便宜
行数: 2274  有效率: 0.4374


...

,timestamp,instrument,factor_value
0,2003-09-10,A,0.586499
1,2003-09-10,A,0.586842
2,2003-09-10,A,0.587369
3,2003-09-10,A,0.587560
4,2003-09-10,A,0.587734
5,2003-09-10,A,0.588466
6,2003-09-10,A,0.588823
7,2003-09-10,A,0.588823
8,2003-09-10,A,0.588659
9,2003-09-10,A,0.588608



=== quotes_zscore_bid_ask_spread ===
描述: 报价数据 - bid/ask 比值截面标准化，反映相对买卖价差（值越高买卖价差越小，流动性越好）
行数: 18386837  有效率: 1.0


,timestamp,instrument,factor_value
0,2003-09-10,A,0.029349
1,2003-09-10,A,0.029349
2,2003-09-10,A,-0.006961
3,2003-09-10,A,-0.006961
4,2003-09-10,A,-0.006961
5,2003-09-10,A,-0.006961
6,2003-09-10,A,-0.006961
7,2003-09-10,A,-0.006961
8,2003-09-10,A,-0.006961
9,2003-09-10,A,-0.006961



=== trades_rank_price ===
描述: 逐笔成交 - 成交价格截面排名，反映股票价格相对高低水平
行数: 10657214  有效率: 1.0


,timestamp,instrument,factor_value
0,2003-09-10,A,0.529851
1,2003-09-10,A,0.529851
2,2003-09-10,A,0.529851
3,2003-09-10,A,0.529851
4,2003-09-10,A,0.529851
5,2003-09-10,A,0.529851
6,2003-09-10,A,0.529851
7,2003-09-10,A,0.529851
8,2003-09-10,A,0.529851
9,2003-09-10,A,0.529851
